# CEG5306 HW1: Robot Battery Policy Iteration

This notebook models a robot's charging and task decisions as a finite Markov decision process (MDP).

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Robot Battery Policy Iteration notebook initialized.")

Robot Battery Policy Iteration notebook initialized.


## 1. MDP Formulation

The state is

$$
s=(L,B),\qquad L\in\{\texttt{charging\_zone},\texttt{task\_zone}\},\qquad B\in\{0,1,2,3,4,5\}.
$$

The state is Markov because the current location and battery level contain all information needed to determine the legal actions, immediate reward, and next state. The initial-state distribution is deterministic:

$$
P\bigl(S_0=(\texttt{charging\_zone},5)\bigr)=1.
$$

All transitions are deterministic. For each legal state-action pair, the transition probability is 1 for exactly one next state and 0 for every other state.

In [2]:
LOCATIONS = ("charging_zone", "task_zone")
BATTERY_LEVELS = tuple(range(6))
STATES = tuple(
    (location, battery)
    for location in LOCATIONS
    for battery in BATTERY_LEVELS
)
STATE_TO_INDEX = {state: index for index, state in enumerate(STATES)}

INITIAL_STATE = ("charging_zone", 5)
GAMMA = 0.95

RECHARGE = "recharge"
TRAVEL_TO_TASK = "travel_to_task"
PERFORM_TASK = "perform_task"
RETURN_TO_CHARGING = "return_to_charging"
EMERGENCY_RESCUE = "emergency_rescue"

In [3]:
def legal_actions(state):
    location, battery = state

    if location == "charging_zone":
        actions = []
        if battery < 5:
            actions.append(RECHARGE)
        if battery >= 1:
            actions.append(TRAVEL_TO_TASK)
        return tuple(actions)

    if battery == 0:
        return (EMERGENCY_RESCUE,)

    return (PERFORM_TASK, RETURN_TO_CHARGING)


def transition(state, action):
    location, battery = state
    if action not in legal_actions(state):
        raise ValueError(f"Illegal action {action!r} for state {state!r}")

    if action == RECHARGE:
        return ("charging_zone", battery + 1), -1.0
    if action == TRAVEL_TO_TASK:
        return ("task_zone", battery - 1), -0.5
    if action == PERFORM_TASK:
        return ("task_zone", battery - 1), 5.0
    if action == RETURN_TO_CHARGING:
        return ("charging_zone", battery - 1), -0.5
    return ("charging_zone", 0), -20.0


assert len(STATES) == 12
assert len(set(STATES)) == 12
assert INITIAL_STATE in STATE_TO_INDEX

for state in STATES:
    actions = legal_actions(state)
    assert actions, f"No legal action for {state}"
    for action in actions:
        next_state, reward = transition(state, action)
        assert next_state in STATE_TO_INDEX
        assert np.isfinite(reward)

print(f"MDP validation passed: {len(STATES)} states")
print(f"Initial state: {INITIAL_STATE}")

MDP validation passed: 12 states
Initial state: ('charging_zone', 5)


## 2. Policy Evaluation and the Value Function

For a fixed deterministic policy $\pi$, the state-value function is

$$
V^\pi(s)=\mathbb{E}\left[\sum_{t=0}^{\infty}\gamma^t r(S_t,\pi(S_t))\mid S_0=s\right].
$$

It satisfies the Bellman expectation equation

$$
V^\pi=r^\pi+\gamma T^\pi V^\pi.
$$

Rearranging gives the linear system

$$
(I-\gamma T^\pi)V^\pi=r^\pi.
$$

We solve this system with `numpy.linalg.solve`. We do not explicitly calculate a matrix inverse.

In [4]:
def evaluate_policy(policy):
    num_states = len(STATES)
    transition_matrix = np.zeros((num_states, num_states))
    reward_vector = np.zeros(num_states)

    for state in STATES:
        row = STATE_TO_INDEX[state]
        action = policy[state]
        next_state, reward = transition(state, action)
        transition_matrix[row, STATE_TO_INDEX[next_state]] = 1.0
        reward_vector[row] = reward

    values = np.linalg.solve(
        np.eye(num_states) - GAMMA * transition_matrix,
        reward_vector,
    )
    return values

## 3. Policy Improvement

For every legal action, one-step lookahead calculates

$$
Q^\pi(s,a)=r(s,a)+\gamma\sum_{s'}P(s'\mid s,a)V^\pi(s').
$$

Because this environment is deterministic, $s_{\mathrm{next}}=f(s,a)$ is unique, so

$$
Q^\pi(s,a)=r(s,a)+\gamma V^\pi(s_{\mathrm{next}}).
$$

The improved policy is greedy with respect to $V^\pi$:

$$
\pi_{\mathrm{new}}(s)=\arg\max_a Q^\pi(s,a).
$$

In [5]:
def action_value(state, action, values):
    next_state, reward = transition(state, action)
    return reward + GAMMA * values[STATE_TO_INDEX[next_state]]


def improve_policy(values):
    improved = {}
    for state in STATES:
        actions = legal_actions(state)
        improved[state] = max(
            actions,
            key=lambda action: action_value(state, action, values),
        )
    return improved

## 4. Policy Iteration

Policy iteration alternates between exact policy evaluation and greedy policy improvement. It stops when a complete improvement step changes no action. At that point, the policy is greedy with respect to its own value function and is therefore optimal.

In [6]:
def policy_iteration():
    policy = {state: legal_actions(state)[0] for state in STATES}
    change_history = []

    while True:
        values = evaluate_policy(policy)
        improved_policy = improve_policy(values)
        changed_states = sum(
            improved_policy[state] != policy[state]
            for state in STATES
        )
        change_history.append(changed_states)

        if changed_states == 0:
            return policy, values, change_history

        policy = improved_policy


optimal_policy, optimal_values, change_history = policy_iteration()
print(f"Policy iteration converged in {len(change_history)} rounds")
print(f"Changed states per round: {change_history}")

Policy iteration converged in 3 rounds
Changed states per round: [3, 2, 0]


In [7]:
final_values = evaluate_policy(optimal_policy)
bellman_update = np.array([
    action_value(state, optimal_policy[state], final_values)
    for state in STATES
])
bellman_residual = np.max(np.abs(final_values - bellman_update))

assert bellman_residual < 1e-10
assert improve_policy(final_values) == optimal_policy
assert optimal_policy[("charging_zone", 4)] == RECHARGE
assert optimal_policy[("charging_zone", 5)] == TRAVEL_TO_TASK
assert optimal_policy[("task_zone", 2)] == PERFORM_TASK
assert optimal_policy[("task_zone", 1)] == RETURN_TO_CHARGING

print(f"Bellman residual: {bellman_residual:.3e}")
print("Policy stability check passed")
print("Expected policy behavior check passed")

Bellman residual: 5.329e-15
Policy stability check passed
Expected policy behavior check passed


## 5. Optimal Value Function and Policy

After convergence, the stable policy is $\pi^*$ and its evaluated state-value function is

$$
V^{\pi^*}=V^*.
$$

The value table reports the expected discounted return from every state. The policy table reports the maximizing action in every state.

In [8]:
short_labels = {
    RECHARGE: "Recharge",
    TRAVEL_TO_TASK: "Go to task",
    PERFORM_TASK: "Do task",
    RETURN_TO_CHARGING: "Return",
    EMERGENCY_RESCUE: "Rescue",
}

value_grid = np.array([
    [optimal_values[STATE_TO_INDEX[(location, battery)]] for battery in BATTERY_LEVELS]
    for location in LOCATIONS
])
policy_grid = np.array([
    [short_labels[optimal_policy[(location, battery)]] for battery in BATTERY_LEVELS]
    for location in LOCATIONS
])

print("Optimal value function V*:")
print(np.round(value_grid, 3))
print("Optimal policy pi*:")
print(policy_grid)

Optimal value function V*:
[[13.103 14.846 16.68  18.61  20.642 22.781]
 [-7.552 11.948 16.351 20.533 24.506 28.281]]
Optimal policy pi*:
[['Recharge' 'Recharge' 'Recharge' 'Recharge' 'Recharge' 'Go to task']
 ['Rescue' 'Return' 'Do task' 'Do task' 'Do task' 'Do task']]


In [9]:
policy_figure, policy_axis = plt.subplots(figsize=(13, 3.2))
policy_axis.axis("off")
policy_table = policy_axis.table(
    cellText=policy_grid,
    rowLabels=("Charging zone", "Task zone"),
    colLabels=[f"Battery {battery}" for battery in BATTERY_LEVELS],
    cellLoc="center",
    loc="center",
)
policy_table.auto_set_font_size(False)
policy_table.set_fontsize(10)
policy_table.scale(1.0, 2.0)
for (row, column), cell in policy_table.get_celld().items():
    if row == 0:
        cell.set_facecolor("#1F4E78")
        cell.set_text_props(color="white", weight="bold")
    elif column == -1:
        cell.set_facecolor("#D9EAF7")
        cell.set_text_props(weight="bold")
    else:
        cell.set_facecolor("#F5F9FC" if row % 2 else "#E8F2F8")
policy_axis.set_title("Optimal Robot Battery Policy", fontsize=15, weight="bold", pad=18)
policy_figure.tight_layout()
policy_figure.savefig(
    OUTPUT_DIR / "optimal_policy.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

/var/folders/lq/3z4hc94164579lwd9_v5v6pc0000gn/T/ipykernel_36790/2078911627.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
value_figure, value_axis = plt.subplots(figsize=(10.5, 3.8))
image = value_axis.imshow(value_grid, cmap="YlGnBu", aspect="auto")
value_axis.set_title("Optimal Value Function $V^*$", fontsize=15, weight="bold", pad=14)
value_axis.set_xlabel("Battery level")
value_axis.set_ylabel("Robot location")
value_axis.set_xticks(range(len(BATTERY_LEVELS)), labels=BATTERY_LEVELS)
value_axis.set_yticks(range(len(LOCATIONS)), labels=("Charging zone", "Task zone"))

threshold = (value_grid.min() + value_grid.max()) / 2
for row in range(value_grid.shape[0]):
    for column in range(value_grid.shape[1]):
        value = value_grid[row, column]
        value_axis.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            color="white" if value > threshold else "#12344D",
            weight="bold",
        )

colorbar = value_figure.colorbar(image, ax=value_axis, pad=0.03)
colorbar.set_label("Expected discounted return")
value_figure.tight_layout()
value_figure.savefig(
    OUTPUT_DIR / "value_heatmap.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

/var/folders/lq/3z4hc94164579lwd9_v5v6pc0000gn/T/ipykernel_36790/2495328979.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Discussion

The state is Markov because location and battery level are sufficient to determine every legal action and next state; no earlier history is needed. Deterministic transitions simplify the Bellman expectation because only one next state has nonzero probability.

Policy Evaluation calculates $V^\pi$ for the current policy by solving its Bellman linear system. Policy Improvement then performs one-step lookahead and selects the action greedy with respect to $V^\pi$. When the policy no longer changes, Bellman's theorem implies that it is optimal, and the final evaluated values satisfy $V^{\pi^*}=V^*$.

The learned behavior is interpretable: the robot recharges to full capacity, travels to the task zone, performs tasks while at least two battery units remain, and returns when only one unit remains. The heatmap shows why: additional battery increases the expected discounted return, while being stranded in the task zone has negative value because emergency rescue is costly.

## Why this submission should be highlighted

This submission shows how policy iteration enables an autonomous robot to balance productive work against charging and travel costs. The final policy and value-function heatmap make the robot's long-term energy-management strategy directly interpretable for every battery level.